# Core figures — cutouts and Afρ

Publication figures for one or several targets. Run `query.ipynb` and
`afrho.ipynb` (or `notebooks/main.py`) first so there are cutouts and photometry
tables on disk.

Every figure uses the project style from `ztfcomet.rcparams` — import it, never
restate the settings. `savefig.dpi` is 200 for one-off figures; drop to **50**
when writing hundreds of files in a batch.

The annotated-cutout plot lived in four separate copies across the old notebooks
and had already drifted between them. It is one function now,
`ztfcomet.plot_cutout`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Make the package importable from a checkout, wherever the kernel started.
_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
              if (p / "ztfcomet" / "__init__.py").is_file()), None)
if _root and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from astropy.io import fits
from astropy.wcs import WCS

import ztfcomet as zc
from ztfcomet import rcparams          # project matplotlib style

zc.setup_logging()
print(zc.directory.describe())

## 1. Load photometry

`load` reads a table written by `run_photometry`. Missing targets are reported
rather than raising, so a partially-reduced set still plots.

In [ ]:
def load(name):
    """Read the photometry table for a target, or None if it is not reduced yet."""
    path = zc.result_dir(name, create=False) / f"photometry_{zc.target_slug(name)}.csv"
    if not path.exists():
        print(f"  {name}: not reduced yet ({path})")
        return None
    table = pd.read_csv(path)
    n_ok = int(table.quality_ok.sum()) if "quality_ok" in table else len(table)
    print(f"  {name}: {len(table)} frames, {n_ok} unflagged")
    return table


TARGETS = ["24P"]        # add more, e.g. ["24P", "240P", "2P"]

tables = {name: t for name in TARGETS if (t := load(name)) is not None}
assert tables, "nothing to plot — run main.py first"

## 2. Single cutout

The full annotated form: WCS axes, N/E compass, anti-solar and anti-velocity
vectors, an ephemeris panel, and the photometric aperture and sky annulus drawn
where they were actually measured.

In [ ]:
name  = TARGETS[0]
table = tables[name]
target = zc.get_target(name)

# Pick the best-SNR unflagged frame.
good = table[table.quality_ok] if "quality_ok" in table else table
row  = good.loc[good.snr.idxmax()]

ax = zc.plot_cutout(zc.data_dir(name) / row.file, row=row, target=target,
                    show_apertures=True, phot_config=target.phot)
ax.figure.savefig(zc.fig_dir(name) / f"cutout_{Path(row.file).stem}.png", dpi=200)
plt.show()

## 3. Contact sheet

A quick look across a whole run — useful for spotting frames where the comet is
near the cutout edge or the field is crowded.

In [ ]:
fig = zc.plot_cutout_grid(good, zc.data_dir(name), target=target,
                          ncols=4, nmax=12, phot_config=target.phot)
fig.savefig(zc.fig_dir(name) / f"cutout_grid_{zc.target_slug(name)}.png", dpi=150)
plt.show()

### Flagged frames

Worth looking at rather than trusting the flags blindly — the flag explains
*why* a frame is suspect, and the picture shows whether the call was right.

In [ ]:
flagged = table[~table.quality_ok] if "quality_ok" in table else table.iloc[:0]

if len(flagged):
    # Show contaminated frames first — they are the ones whose Afrho is inflated.
    order = flagged.sort_values("contam_ratio", ascending=False) \
        if "contam_ratio" in flagged else flagged
    show = order.head(4)

    fig = plt.figure(figsize=(4.6 * len(show), 4.6))
    for i, (_, row) in enumerate(show.iterrows(), start=1):
        path = zc.data_dir(name) / row.file
        if not path.exists():
            continue
        with fits.open(path) as hdul:
            wcs = WCS(hdul[0].header)
        reasons = [c.replace("flag_", "") for c in zc.FLAG_COLUMNS if row.get(c)]
        zc.plot_cutout(path, row=row, target=target, annotate=False,
                       phot_config=target.phot,
                       ax=fig.add_subplot(1, len(show), i, projection=wcs),
                       title="\n".join(reasons))
    fig.tight_layout()
    fig.savefig(zc.fig_dir(name) / f"flagged_{zc.target_slug(name)}.png", dpi=150)
    plt.show()
else:
    print("no flagged frames")

## 4. Batch cutouts

Hundreds of PNGs at 200 dpi is slow and large; `save_all_cutouts` defaults to
`dpi=50` for exactly this case.

In [ ]:
# paths = zc.save_all_cutouts(table, zc.data_dir(name), zc.fig_dir(name) / "cutout",
#                             target=target, phot_config=target.phot, dpi=50)
# print(f"{len(paths)} figures")

## 5. Afρ lightcurve — single target

Flagged points are drawn as **open** symbols rather than removed, so nothing
disappears from a figure without being visible. Pass `only_good=True` to drop
them once you are satisfied the flags are right.

In [ ]:
bands = [b for b in ("ZTF_g", "ZTF_r", "ZTF_i") if (table["filter"] == b).any()]

ax = zc.plot_afrho({name: table}, filters=bands, x="rh")
ax.set_title(f"{name}   " + r"$\rho$ = " + f"{target.phot.rho_km:.0f} km")
ax.figure.savefig(zc.fig_dir(name) / f"afrho_rh_{zc.target_slug(name)}.png", dpi=200)
plt.show()

In [ ]:
# Versus time from perihelion, where the perihelion time is known.
if target.perihelion_jd:
    label, tp = list(target.perihelion_jd.items())[-1]
    ax = zc.plot_afrho({name: table}, filters=bands, x="tp",
                       perihelion_jd={name: tp})
    ax.set_title(f"{name}   " + r"$T_\mathrm{p}$ = " + f"{label}")
    ax.axvline(0, color="0.6", ls=":", lw=1.5)
    ax.figure.savefig(zc.fig_dir(name) / f"afrho_tp_{zc.target_slug(name)}.png", dpi=200)
    plt.show()
else:
    print(f"no perihelion time configured for {name}; add one in ztfcomet/config.py")

## 6. Multiple targets

One panel, one colour per target, one marker per band.

In [ ]:
if len(tables) > 1:
    ax = zc.plot_afrho(tables, filters=["ZTF_r"], x="rh")
    ax.set_title(r"$A(0\degree)f\rho$ — all targets (ZTF_r)")
    ax.figure.savefig(zc.fig_dir(None) / "afrho_all_targets.png", dpi=200)
    plt.show()
else:
    print("add more targets to TARGETS above for the comparison plot")

### Colour: Afρ(g) versus Afρ(r)

Same night, both bands — a useful consistency check, and a dust-colour
diagnostic in its own right.

In [ ]:
if {"ZTF_g", "ZTF_r"} <= set(table["filter"].unique()):
    night = np.floor(table.obsjd - 0.5).astype(int)
    pair = (table.assign(night=night)
                 .query("quality_ok")
                 .groupby(["night", "filter"])
                 .agg(afrho=("afrho0_cm", "median"), rh=("r", "median"))
                 .unstack("filter"))
    pair = pair.dropna(subset=[("afrho", "ZTF_g"), ("afrho", "ZTF_r")])

    if len(pair):
        fig, ax = plt.subplots(figsize=(7, 6))
        s = ax.scatter(pair[("afrho", "ZTF_r")], pair[("afrho", "ZTF_g")],
                       c=pair[("rh", "ZTF_r")], cmap="viridis", s=60,
                       edgecolor="k", lw=0.5)
        lims = [0, max(pair[("afrho", "ZTF_r")].max(), pair[("afrho", "ZTF_g")].max()) * 1.1]
        ax.plot(lims, lims, "k--", lw=1.5, label="1:1")
        ax.set(xlim=lims, ylim=lims,
               xlabel=r"$A(0\degree)f\rho$ (cm), ZTF_r",
               ylabel=r"$A(0\degree)f\rho$ (cm), ZTF_g")
        fig.colorbar(s, ax=ax, label=r"$r_\mathrm{h}$ (AU)")
        ax.legend(frameon=False)
        fig.savefig(zc.fig_dir(name) / f"afrho_color_{zc.target_slug(name)}.png", dpi=200)
        plt.show()
    else:
        print("no nights with both bands")
else:
    print("need both ZTF_g and ZTF_r frames")

## 7. Diagnostics

Not for publication — for deciding whether the numbers above can be trusted.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))

axes[0, 0].scatter(table.r, table.rho_fwhm, c="k", s=25)
axes[0, 0].axhline(target.phot.min_rho_fwhm, color="r", ls="--",
                   label=f"threshold = {target.phot.min_rho_fwhm}")
axes[0, 0].set(xlabel=r"$r_\mathrm{h}$ (AU)", ylabel=r"$\rho$ / FWHM")
axes[0, 0].legend(fontsize=11, frameon=False)

axes[0, 1].scatter(table.edge_dist_pix, table.sky_out_pix, c="k", s=25)
lim = np.nanmax([table.edge_dist_pix.max(), table.sky_out_pix.max()]) * 1.05
axes[0, 1].plot([0, lim], [0, lim], "r--", label="annulus hits the edge")
axes[0, 1].set(xlabel="distance to array edge (px)", ylabel="sky annulus $r_{out}$ (px)",
               xlim=(min(0, table.edge_dist_pix.min() * 1.1), lim))
axes[0, 1].legend(fontsize=11, frameon=False)

axes[0, 2].scatter(table.filter_mag, table.snr, c="k", s=25)
axes[0, 2].axhline(3, color="r", ls="--", label="SNR = 3")
axes[0, 2].set(xlabel="calibrated mag", ylabel="SNR", yscale="log")
axes[0, 2].legend(fontsize=11, frameon=False)

axes[1, 0].hist(table.centroid_shift_pix.dropna(), bins=25, color="0.6", edgecolor="k")
axes[1, 0].set(xlabel="centroid shift from ephemeris (px)", ylabel="frames")

# Contamination: G_eff against the comet's predicted magnitude. Points below
# the line carry >= contam_flux_ratio of the comet flux.
if "contam_g_eff" in table:
    finite = table[np.isfinite(table.contam_g_eff)]
    if len(finite):
        axes[1, 1].scatter(finite.tmag, finite.contam_g_eff,
                           c=np.where(finite.flag_contaminated, "r", "k"), s=30)
        span = np.array([finite.tmag.min() - 0.5, finite.tmag.max() + 0.5])
        offset = -2.5 * np.log10(target.phot.contam_flux_ratio)
        axes[1, 1].plot(span, span + offset, "r--",
                        label=f"{target.phot.contam_flux_ratio:.0%} of comet flux")
        axes[1, 1].invert_yaxis()
        axes[1, 1].set(xlabel=r"comet $T_\mathrm{mag}$", ylabel=r"$G_\mathrm{eff}$ in aperture")
        axes[1, 1].legend(fontsize=11, frameon=False)
    else:
        axes[1, 1].text(0.5, 0.5, "no Gaia sources\nin any aperture", ha="center",
                        va="center", transform=axes[1, 1].transAxes)
        axes[1, 1].set_axis_off()

    axes[1, 2].scatter(table.contam_ratio.clip(1e-3), table.afrho0_cm,
                       c=np.where(table.flag_contaminated, "r", "k"), s=30)
    axes[1, 2].axvline(target.phot.contam_flux_ratio, color="r", ls="--")
    axes[1, 2].set(xlabel="contaminant / comet flux", ylabel=r"$A(0\degree)f\rho$ (cm)",
                   xscale="log")

fig.suptitle(f"{name} — diagnostics")
fig.tight_layout()
fig.savefig(zc.fig_dir(name) / f"diagnostics_{zc.target_slug(name)}.png", dpi=150)
plt.show()